# 05 - Retrieval
## Shakespeare Evidence Desk


## Why This Stage

*In general — what retrieval is and why any RAG system needs it.* Everything up to now ran offline and never saw a question: parsing, chunking, embedding, and indexing prepared the corpus blind. Retrieval is where that preparation finally pays off — it takes a user's query and returns the handful of passages most likely to answer it. It is the hinge between the offline index and the online answer, and it quietly sets the ceiling for everything downstream: the generator can only quote what retrieval surfaces, so a passage missed here is lost for good, however good the model is.

The core decision retrieval owns is *how to decide a passage is relevant to a query* — by the literal words it contains, by its meaning, or by a fusion of both — together with how many candidates to return and which hard filters to apply. These choices trade recall (don't miss the right passage) against precision and cost (don't flood the next stage with noise). Because different methods fail on different queries, retrieval is compared and measured rather than assumed — and, unlike the offline stages, its quality can finally be graded against the labeled questions directly.


- **Goal:** turn a user's question into a short, ranked list of candidate passages from the index — the first **online** stage, and the recall ceiling for every answer that follows. A passage not retrieved here can never be reranked, assembled, or cited.
- **Loads from Step 04:** `.artifacts/index.json` (the searchable index: every row's vector + provenance, plus the IVF structure) and `.artifacts/indexing-handoff.json` (the index decision). The labeled questions (`.artifacts/eval-questions.json`) grade retrieval and supply filter scopes.
- **The experiment:** compare three retrieval methods on the same index and questions — **keyword/BM25** (match literal words), **dense** (match meaning, reusing the stage-04 index), and **hybrid** (fuse both with reciprocal rank fusion) — then settle **top-k** and demonstrate **metadata filters**.
- **Runs online:** the only network call is embedding the labeled *questions* with the same keyless Azure deployment and on-disk cache as notebooks 03/04; already-embedded questions are served from cache with zero API calls. BM25 and fusion are pure local compute.
- **Preserve provenance:** candidates carry their `chunk_id`, work, scenes, and speakers through unchanged, so reranking, context building, and citations downstream still resolve back to the source lines.
- **Not later stages:** this notebook does not rerank (06), rewrite the query (09), build context (07), or answer (08). It exports the selected retrieval configuration for notebook 06 (reranking).


## Conceptual Foundations

### Retrieval is where the query finally meets the corpus

Everything before this stage ran *offline* and *query-blind*: parsing, chunking, embedding, and indexing all prepared the corpus without ever seeing a question. Retrieval is the first **online** stage — the first time a user's query exists — and its one job is to turn that query into a short, ranked list of candidate passages the later stages can rerank (06), assemble (07), and quote (08). Everything downstream can only work with what retrieval returns: **a passage not retrieved can never be cited**, no matter how good the generator is. Retrieval sets the recall ceiling for the whole answer.

### Two ways to match a query: words vs. meaning

There are two fundamentally different ways to decide a passage is relevant:

- **Lexical / sparse (keyword, BM25):** match on the literal *words*. A passage scores high if it contains the query's terms, weighted so rare words count more and long passages don't win just by being long. Strengths: exact quotes, proper names, rare tokens ("hebenon", "Birnam"). Blind spot: vocabulary mismatch — it cannot connect "killed" to "slew" or "poison" to "envenom'd".
- **Semantic / dense (vector):** match on *meaning*. Embed the query into the same vector space as the chunks (notebook 03) and return the nearest vectors. Strengths: paraphrase, synonyms, conceptual questions. Blind spot: it can miss an exact rare token, and it can return something *topically* close but not literally what was asked.

Neither dominates. They fail on **different** queries, which is exactly why fusing them can beat either alone.

### Hybrid retrieval and reciprocal rank fusion (RRF)

**Hybrid** retrieval runs both a sparse and a dense search and *fuses* their rankings. The robust, score-agnostic way to fuse is **reciprocal rank fusion (RRF)**: ignore the raw scores (BM25 and cosine live on incomparable scales) and use only each passage's **rank** in each list. A passage's fused score is $\sum_i \frac{1}{k + \text{rank}_i}$ summed over the lists it appears in, where $k$ (a small constant, here 60) damps the influence of the very top ranks so a strong second-place in both lists can beat a first-place in only one. RRF needs no score normalization and no tuning per corpus, which is why it is the common production default.

### The knobs retrieval owns

- **top-k** — how many candidates to return. Too small risks dropping the one relevant passage (hurts recall); too large passes noise to reranking and inflates cost. It is a recall-vs-precision dial, retuned once reranking (06) and generation (08) can measure its real effect.
- **filters** — hard metadata constraints applied *before or during* search (e.g. `work_id = Hamlet`). A filter is not ranking — it removes non-matching rows entirely. Filters raise precision and cut cost when the query names a scope, but a wrong filter can remove the answer, so filter *correctness* is measured, not assumed.
- **fusion constant (`k`)** — RRF's damping term; fixed here so we compare retrieval *methods* fairly, one change at a time.

### What retrieval is *not* (clean stage boundaries)

- **Not reranking (06):** retrieval returns a cheap, broad candidate set; reranking reorders a *fixed* candidate set with a more expensive model. Keeping them separate lets each be measured alone.
- **Not query rewriting (09):** retrieval here takes the user's question *as written*. Rewriting, decomposition, and multi-query live in notebook 09, which wraps this stage.
- **Not generation (08):** retrieval never writes an answer; it only supplies evidence. A high Recall@k here is necessary but not sufficient for a correct, well-cited answer.

### Common misconceptions and failure modes

- **"Dense always beats keyword."** False for exact quotes and rare names — the Evidence Desk's bread and butter — where BM25 often wins. The honest answer is query-dependent, which motivates hybrid.
- **"A bigger top-k is safer."** It raises recall but floods reranking and generation with noise and cost; precision and latency pay for it.
- **"Filters are free precision."** Only when the scope is correct. A filter that excludes the answer's work drops recall to zero for that query — which is why we measure filter correctness and empty-result rate.
- **"Retrieval metrics prove the system works."** Recall@k and MRR here are *proxies* on a small labeled set. Whether retrieval helps the final answer is judged end-to-end in generation (08) and continuous evaluation (10).


## 1. Load the Previous Selection

- **Purpose:** start from notebook 04's exported index and decision record, not hidden kernel state. Retrieval only *queries* the index — it does not rebuild it — so the index type, metric, ANN parameters, and metadata schema are fixed input here.
- **What the code does:** locates the repository root, reads `index.json` (the rows with vectors + provenance, and the IVF structure), reconstructs the stage-04 **dense search** exactly (IVF centroids/lists at the selected `nprobe`, or exact scan if that was chosen), and builds the **answer key** from the labeled questions (reusing the citation parser from notebooks 02–04) so Section 4 can grade retrieval and derive filter scopes.
- **Fails loudly, early:** if `.artifacts/index.json` is absent, run notebook 04 first. If the index was built on offline-fallback vectors (`embedding_backend != 'hosted'`), the notebook **stops here** — retrieving against fake vectors is meaningless.
- **Expected result:** the index type and metric, the row count and stored dimension, the (hosted) embedding backend, and how many labeled questions resolved to an indexed scene the grader can use. No network call happens here.


In [ ]:
import json
import math
import re
from collections import defaultdict
from pathlib import Path

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
index_path = ARTIFACTS / 'index.json'
indexing_handoff_path = ARTIFACTS / 'indexing-handoff.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not index_path.is_file():
    raise RuntimeError('Missing .artifacts/index.json. Run notebook 04 first.')

index_payload = json.loads(index_path.read_text(encoding='utf-8'))
indexing_handoff = json.loads(indexing_handoff_path.read_text(encoding='utf-8')) if indexing_handoff_path.is_file() else {}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

index_type = index_payload['index_type']          # 'ivf' or 'exact', the stage-04 selection
metric = index_payload['metric']                  # 'inner_product' (unit vectors, so dot == cosine)
stored_dims = index_payload['dims']
normalized = index_payload.get('normalized', True)
embedding_backend = index_payload.get('embedding_backend', 'unknown')
records = index_payload['records']                # chunk_id, work_id, work, scenes, speakers, text, embedding

# Stop now if the upstream vectors are not real: retrieving against offline-fallback vectors is meaningless.
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"index.json was built on the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebooks 03 and 04 against Azure before retrieving.")

vectors = [r['embedding'] for r in records]
N = len(vectors)
work_titles = {r['work_id']: r['work'] for r in records}


def dot(a, b):
    # Score between two unit vectors; equals cosine similarity. Higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


# ---- reconstruct stage 04's dense search exactly, so retrieval queries the index we actually selected ----
def exact_search_rows(query_vector, candidate_rows=None):
    pool = range(N) if candidate_rows is None else candidate_rows
    return sorted(pool, key=lambda i: dot(query_vector, vectors[i]), reverse=True)


if index_type == 'ivf':
    ivf = index_payload['ivf']
    ivf_centroids = ivf['centroids']
    ivf_lists = {int(cluster): rows for cluster, rows in ivf['lists'].items()}
    nprobe = index_payload['params']['nprobe']

    def dense_search(query_vector, k, candidate_rows=None):
        # Open only the nprobe nearest clusters (the stage-04 dial), then exact-score within that pool.
        nearest = sorted(range(len(ivf_centroids)),
                         key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
        pool = [i for c in nearest for i in ivf_lists.get(c, ())]
        if candidate_rows is not None:
            allowed = set(candidate_rows)
            pool = [i for i in pool if i in allowed]
        return exact_search_rows(query_vector, pool)[:k]
else:
    nprobe = None

    def dense_search(query_vector, k, candidate_rows=None):
        return exact_search_rows(query_vector, candidate_rows)[:k]

# ---- answer key: turn each labeled question into the set of index rows holding its cited scene ----
ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(value):
    # Normalize a scene number written as a Roman numeral or digits ('V' or '5' -> 5).
    if value is None:
        return None
    text = str(value).strip().upper()
    if text.isdigit():
        return int(text)
    total, previous = 0, 0
    for char in reversed(text):
        if char not in ROMAN_VALUES:
            return None
        current = ROMAN_VALUES[char]
        total += -current if current < previous else current
        previous = max(previous, current)
    return total or None


lowercase_title = {work_id: title.lower() for work_id, title in work_titles.items()}


def cited_locations(question):
    # Read the label's citation -> list of (work, act, scene); uses the label only, never the question's words.
    label_text = (question.get('question', '') + ' ' + question.get('expected_evidence', '')).lower()
    works = [work_id for work_id, title in lowercase_title.items() if title in label_text]
    act_scene_pairs = re.findall(r'act\s+(\d+|[ivxlc]+)\s*,?\s*scene\s+(\d+|[ivxlc]+)',
                                 question.get('expected_evidence', '').lower())
    acts_scenes = [(roman_to_int(act), roman_to_int(scene)) for act, scene in act_scene_pairs]
    return [(work_id, act, scene) for work_id in works for act, scene in acts_scenes]


def row_in_scene(record, work_id, act, scene):
    return record['work_id'] == work_id and any(a == act and s == scene for a, s in record['scenes'])


def works_for_question(question):
    # The index work_ids named by the question's 'expected_works' titles — used as a metadata filter scope.
    wanted = {title.lower() for title in question.get('expected_works', [])}
    return {work_id for work_id, title in lowercase_title.items() if title in wanted}


# For each answerable question: the relevant row set (graded truth) and the work scope (filter demo).
answer_key = []
for question in evaluation_questions.get('development', []):
    if not question.get('answerable'):
        continue
    locations = [(w, a, s) for w, a, s in cited_locations(question) if a is not None and s is not None]
    relevant_rows = {i for i, record in enumerate(records)
                     for (w, a, s) in locations if row_in_scene(record, w, a, s)}
    if relevant_rows:
        answer_key.append((question, relevant_rows, works_for_question(question)))

print(f'Loaded {N:,} indexed rows ({index_type}, metric={metric}, {stored_dims} dims) from the {embedding_backend} backend.')
if index_type == 'ivf':
    print(f'Dense search reproduces stage 04: IVF with nprobe={nprobe} over {len(ivf_centroids)} clusters.')
else:
    print('Dense search reproduces stage 04: exact scan over all rows.')
print(f'Answer key: {len(answer_key)} labeled dev question(s) resolved to an indexed scene.')


## 2. Explain the Alternatives

A retrieval method is **good** when, for a labeled question, it ranks a passage from the cited scene at or near the top — across *both* exact-quote questions (where literal words matter) and conceptual ones (where meaning matters). We **fix the index** (the stage-04 vectors, metric, and ANN config) and the question set, and vary only the *matching strategy* and its knobs.

### What we actually compare vs. what we only explain

You asked whether TF‑IDF, semantic search, hybrid, metadata filtering, etc. are all being compared. The honest answer — stated up front so the experiment is not mistaken for the whole field:

| Method | Role here | Why |
|---|---|---|
| **BM25** (keyword / lexical) | ✅ **compared** (run + measured) | the standard sparse baseline |
| **Dense** (semantic / vector) | ✅ **compared** (run + measured) | the standard dense baseline |
| **Hybrid (RRF)** | ✅ **compared** (run + measured) | fuses the two above |
| **Metadata filtering** | ◐ **demonstrated + checked** (not a competing arm) | it is a *constraint* layered on any method, not a rival ranker, so we show it works rather than race it |
| **TF‑IDF** | ○ **explained, not run** | BM25 is its strictly better successor (adds term saturation + length normalization), so running plain TF‑IDF would only lose |
| **SPLADE, ColBERT, multi‑vector** | ○ **explained, not run** | need a different index (learned-sparse or token-level) than stage 04 built — out of scope here |
| **HyDE / query rewriting** | ○ **explained → notebook 09** | they change the *query*, not the ranking; measured where they belong |
| **Reranking (cross-encoder / LLM)** | ○ **explained → notebook 06** | reorders a *fixed* candidate set; the next stage |

So three true alternatives are compared; the rest are explained so the choice makes sense, with a pointer to where (if anywhere) they are actually run.

### How each method actually works (a line or two)

- **TF‑IDF** — score a passage by **term frequency × inverse document frequency**: a query word counts more the more often it appears in *this* passage (TF) and the rarer it is across *all* passages (IDF). Simple, but it lets a very long passage rack up score by sheer repetition.
- **BM25 (what we run for keyword)** — the modern fix to TF‑IDF. Same TF×IDF idea, plus two corrections: a word's contribution **saturates** (the 10th occurrence adds far less than the 2nd, controlled by `k1`) and the score is **normalized by passage length** (so long passages don't win just by being long, controlled by `b`). Net effect: rank by how many *rare, distinctive* query words a passage contains, fairly across lengths. Great for exact quotes and proper names; blind to synonyms.
- **Dense / semantic search (what we run for dense)** — embed the query into the **same vector space** as the chunks (notebook 03) and return the **nearest** vectors by cosine similarity. It matches *meaning*, so it links "killed" to "slew" — but can miss an exact rare token the query hinges on.
- **Hybrid + RRF (what we run for hybrid)** — run **both** the keyword and the dense search, then **fuse their two ranked lists by position** (reciprocal rank fusion): each passage scores `1/(k+rank)` summed across the lists it appears in. A passage that both methods like rises to the top; no score-scale mixing needed.
- **Metadata filtering** — before ranking, **drop every row that fails a hard condition** on a stored field (here `work_id = Hamlet`). It is not a ranker; it narrows *which* rows any of the three methods is allowed to return. Correct scope = free precision; wrong scope = the answer silently deleted.

**The choices, in easy terms:**

| Choice | What it means (plain) | Why it matters / expected failure |
|---|---|---|
| **BM25 / keyword** *(candidate)* | Score a passage by the query's literal **words**, weighting rare words higher and not rewarding long passages just for length. | Wins on exact quotes and rare names ("hebenon", "Birnam"). **Fails** on vocabulary mismatch — can't link "killed" to "slew". |
| **Dense / vector** *(candidate)* | Embed the query and return the **nearest** chunk vectors (the stage-04 index). Matches **meaning**. | Wins on paraphrase and concept questions. **Fails** to pin an exact rare token; can return topically-near-but-wrong passages. |
| **Hybrid (RRF)** *(candidate)* | Run **both**, then fuse by **rank** (reciprocal rank fusion), not raw score. | Recovers each method's misses — a passage strong in either list rises. **Cost:** runs two searches; can *dilute* a case where one method is decisively right by averaging in the other's weaker ranking. |
| **top-k** *(settled here)* | How many candidates to return. | Small-k risks dropping the answer (recall); large-k floods reranking with noise and cost. A dial, retuned once 06/08 can measure it. |
| **Metadata filter** *(demonstrated)* | A hard constraint (`work_id = Hamlet`) that removes non-matching rows before ranking. | Raises precision and cuts cost when the query names a scope — but a **wrong** filter deletes the answer, so correctness is measured. |
| **Fusion constant `k`** *(fixed)* | RRF's damping term (60). | Fixed so we compare *methods* fairly, one change at a time. |

- **Why method is the experiment we actually run:** sparse, dense, and hybrid have a genuine, measurable tradeoff on *this* corpus's mix of exact-quote and conceptual questions. top-k and the filter are settled/demonstrated, not swept as competing arms.
- **Why RRF fuses on rank, not score.** BM25 scores (unbounded, corpus-dependent) and cosine scores (in $[-1, 1]$) live on **incomparable scales** — adding or averaging them is meaningless without fragile per-corpus normalization. RRF sidesteps this: it reads only each passage's *position* in each list, so a passage ranked #2 by BM25 and #3 by dense gets a principled fused score with **no tuning**. That robustness is why it is a common production default.
- **Honest expectation (no pre-ordained winner):** on a tiny labeled set the methods can tie at the top, and a single strong method (often dense) may already hold every question category — in which case the robustness-first rule in Section 5 selects *it*, not hybrid. We still run all three to show *where* each wins by category and to fix the configuration retrieval will serve, while recording hybrid/RRF as the production-robust default to re-confirm as the query mix broadens.

### Where each choice lives in the code

| Choice | Implemented by | Measured / checked by | Status |
|---|---|---|---|
| **BM25 / keyword** | `BM25.search` — Section 3 | the method comparison — Section 4 | implemented **+ measured** |
| **Dense / vector** | `dense_search` (reconstructed from the index) — Section 1 | the method comparison — Section 4 | implemented **+ measured** |
| **Hybrid (RRF)** | `reciprocal_rank_fusion` — Section 3 | the method comparison — Section 4 | implemented **+ measured** |
| **top-k** | `TOP_K` — Section 3 | recall@k / MRR at that k — Section 4 | settled |
| **Metadata filter** | `rows_allowed_by_filter` applied in every retriever — Section 3 | filter correctness + empty-result rate — Section 4 | demonstrated **+ checked** |
| **Fusion constant `k`** | `RRF_K` — Section 3 | held fixed | settled |
| **Query rewriting** | — none (query used as written) | — none | **deferred to notebook 09** |


### How these terms relate (clearing up a common mix-up)

It is easy to tangle these words together. The clean hierarchy is **two techniques, each with ingredients inside it** — plus one way to combine them:

```
Retrieval techniques
├─ Keyword / lexical search  ── matches literal WORDS
│     ├─ TF-IDF      = term frequency × inverse document frequency (IDF)
│     └─ BM25        = TF-IDF idea + term saturation + length normalization   ← we run this
│           └─ IDF, TF, saturation, length-norm are INGREDIENTS inside BM25
└─ Semantic / vector search ── matches MEANING
      └─ "vector search" and "semantic search" are two names for the SAME thing  ← we run this
            └─ no IDF here; it works on embeddings, not word counts

Hybrid = run BOTH and fuse (RRF)  ← combining them is what "makes it better"
```

So, to correct the common phrasing:

- **BM25 _is_ keyword search** — keyword search doesn't "make BM25 better"; BM25 is a keyword-search algorithm.
- **IDF is not a separate booster** — it's a *part of* BM25 (and TF‑IDF). It's one reason keyword search is smart (rare words weigh more), but it lives *inside* the technique.
- **Vector search = semantic search** — same technique, two names; it uses embeddings, not IDF.
- **What actually "makes retrieval better" is hybrid** — running keyword (BM25) *and* semantic (vector) together, because each covers the other's blind spot (BM25 nails exact words/names; vectors nail paraphrase/meaning).


## Retrieval Strategies: The Landscape and How to Choose

The three arms we run (BM25, dense, hybrid) are the core of modern retrieval, but they sit inside a wider menu. This section maps that menu, then gives a decision guide and the tools that implement it — so the focused experiment below is understood as a deliberate subset, not the whole field.

Legend: **[IMPLEMENTED]** run here · **[EXPLAINED]** common alternative · **[DEFERRED]** belongs to a later stage.

### The families of retrieval

| Family | How it decides relevance | In this notebook? | Where it belongs |
|---|---|---|---|
| **Sparse / lexical** (BM25, TF‑IDF, SPLADE) | literal word overlap, rare words weighted up | ✅ `bm25` | here — the keyword arm |
| **Dense / semantic** (bi-encoder vectors + ANN) | nearness in embedding space (meaning) | ✅ `dense` | here — the vector arm |
| **Hybrid** (sparse + dense, fused by RRF or weighted) | both, combined | ✅ `hybrid-rrf` | here — candidate + production-robust default |
| **Filtered / faceted** (metadata constraints) | hard pre-conditions (`work_id`, date, author) | ◐ demonstrated | here — precision + scope control |
| **Multi-vector / late-interaction** (ColBERT) | token-level matching, more precise but heavier | ✖ | advanced indexing+retrieval redesign |
| **Query transformation** (rewrite, HyDE, decomposition, multi-query) | change the *query* before searching | ✖ **[DEFERRED]** | **notebook 09** (query handling) |
| **Reranking** (cross-encoder, LLM, semantic) | reorder a *fixed* candidate set with a stronger model | ✖ **[DEFERRED]** | **notebook 06** (reranking) |
| **Small-to-big / parent-child** | retrieve a precise child, return its larger parent | ✖ | retrieval architecture, revisit with 04/07 |

### A line on the ones we *don't* run (so the table makes sense)

- **TF‑IDF** — BM25's simpler ancestor (term frequency × inverse document frequency, no saturation or length normalization). We run BM25 instead because it strictly improves on it.
- **SPLADE** — a *learned* sparse model: a transformer predicts which (possibly non-literal) terms a passage should match, giving keyword-style indexing with some semantic reach. Needs its own learned-sparse index, so it is out of scope for the stage-04 vector index.
- **ColBERT / multi-vector** — stores **one vector per token** and scores by best token-to-token matches ("late interaction"); more precise but far heavier to index and store. A different indexing design than this project chose.
- **HyDE** — "Hypothetical Document Embeddings": ask an LLM to *draft a fake answer*, embed that, and search with it — a **query transformation**, so it lives in notebook 09, not here.
- **Reranking** — a cross-encoder or LLM re-scores the *top candidates retrieval already returned*; it refines order, it does not search the corpus. That is notebook 06.
- **Small-to-big / parent-child** — index small precise chunks but return their larger parent for context; an indexing+retrieval architecture to revisit with notebooks 04/07.

### A decision guide

Retrieval choice follows from two questions: *what do your queries look like?* and *do they name a scope?*

```
What dominates your query mix?
├─ Exact quotes, codes, proper names (literal tokens)
│     → start with BM25; add dense for the conceptual tail  → hybrid
├─ Paraphrase, concepts, "how/why" questions (meaning)
│     → start with dense; add BM25 to catch rare literals   → hybrid
└─ A mix of both (our Evidence Desk: exact-quote AND comparison questions)
      → hybrid (RRF) is the robust default to re-confirm as the query mix grows

Does the query name a scope (one work, author, date range)?
├─ Yes → apply a metadata filter (precision + cost win), but verify the scope is right
└─ No  → search the whole corpus; let ranking do the work
```

**Rules of thumb:**
- **Hybrid + RRF is the safe default** when the query mix is mixed or unknown — it rarely loses badly to the best single method and often wins, with no score-normalization tuning. On a small, clean labeled set its edge may not yet show, so measure before committing.
- **Treat top-k as a tunable**, not a constant: retrieve a little wider than the generator needs, let reranking (06) tighten it.
- **Use filters when the scope is explicit and reliable**; never let a filter silently remove the answer — measure its correctness.
- **Don't fold reranking or query rewriting in here** — keep stage boundaries clean so each lever is measured alone (06 and 09).

### Tools and libraries

We implement BM25 and RRF inline (plain Python) so every scoring and fusion rule is visible. Production systems commonly use:

- **Azure AI Search** *(our production target)* — one service offering keyword (BM25), vector (HNSW), **and native hybrid** search with metadata filters and optional semantic ranking, so the three arms here map to one managed query.
- **Elasticsearch / OpenSearch** — mature BM25 plus kNN vector search and hybrid pipelines.
- **Vector DBs with hybrid** — Weaviate, Qdrant, Milvus, Pinecone expose sparse+dense fusion.
- **`rank_bm25`** — a drop-in BM25 for Python prototypes (same Okapi BM25 formula implemented here).
- **LangChain / LlamaIndex** — `EnsembleRetriever` / `QueryFusionRetriever` wrap exactly this sparse+dense RRF pattern.

**The engineering point:** these tools save boilerplate but implement the *same* three ideas taught here — lexical match, semantic match, and rank fusion. Understanding why RRF fuses on rank (not score), and when a filter helps vs. harms, matters more than any one API.


## 3. Run Controlled Experiments

- **Fixed upstream:** the index (stage-04 vectors, metric, IVF config), the stored dimension, the question set, `top-k`, and the RRF constant are held constant; only the retrieval *method* changes between candidates.
- **What the code does:** builds a pure-Python **BM25** lexical index over the chunk text (tokenize, document frequencies, average length), defines **dense** search by reusing the stage-04 index reconstructed in Section 1, and fuses the two rankings with **reciprocal rank fusion**. A `work_filter` argument threads through all three so the same methods can run corpus-wide or restricted to a scope.
- **Query embedding:** the labeled *questions* are embedded with the same keyless Azure deployment and on-disk cache as notebooks 03/04, then truncated + L2-normalized to the stored dimension so a query vector lives in the indexed vectors' space. Because 03/04 already embedded these questions, the cache usually serves them with **0 API calls**.
- **No offline fallback — fail fast:** BM25 and fusion are pure local compute. Question embedding reads the on-disk cache first; if a question is *not* cached and Azure is unreachable, `embed_query` raises and the notebook stops with the Azure error instead of fabricating a query vector.
- **Expected result:** the BM25 vocabulary size, the `top-k` and RRF constant in use, the three registered retrieval methods, and the count of questions embedded for grading (with how many needed a live Azure call).


In [ ]:
import hashlib
import time

from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

TOP_K = 5          # candidates returned per query; the recall/precision dial, retuned once 06/08 can measure it
RRF_K = 60         # reciprocal rank fusion damping constant; fixed so we compare methods, not fusion tuning
CANDIDATE_POOL = max(TOP_K, 20)   # retrieve a wider pool per method before fusing / cutting to TOP_K

_WORD_PATTERN = re.compile(r"[a-z0-9']+")   # a "word" is a run of letters/digits/apostrophes


def tokenize(text):
    """Lowercase the text and split it into word tokens — the unit BM25 counts and matches on."""
    return _WORD_PATTERN.findall(text.lower())


class BM25:
    """A small, transparent Okapi BM25 keyword index over the passage texts.

    BM25 ranks a passage by how many of the query's literal words it contains, with two corrections
    over naive word-counting: rarer words count for more (IDF), and each word's contribution both
    saturates (repeats matter less and less) and is normalized by passage length (so a long passage
    cannot win by sheer bulk). Build the index once, then call `search` for any query.
    """

    def __init__(self, documents, term_saturation=1.5, length_strength=0.75):
        # term_saturation (classic "k1"): how quickly repeated occurrences of a word stop adding score.
        # length_strength (classic "b"): how strongly long passages are penalized (0 = not at all, 1 = fully).
        self.k1 = term_saturation
        self.b = length_strength
        self.tokenized_docs = [tokenize(text) for text in documents]
        self.doc_lengths = [len(tokens) for tokens in self.tokenized_docs]
        self.avg_len = (sum(self.doc_lengths) / len(self.tokenized_docs)) if self.tokenized_docs else 0.0

        # Per passage: how many times each word occurs. Across the corpus: in how many passages each word appears.
        self.term_counts_per_doc = []
        document_frequency = defaultdict(int)
        for tokens in self.tokenized_docs:
            term_counts = defaultdict(int)
            for term in tokens:
                term_counts[term] += 1
            self.term_counts_per_doc.append(term_counts)
            for term in term_counts:
                document_frequency[term] += 1

        # Inverse document frequency: rare words get a higher weight; the +0.5 smoothing keeps it well-behaved.
        total_documents = len(self.tokenized_docs)
        self.idf = {
            term: math.log(1 + (total_documents - doc_count + 0.5) / (doc_count + 0.5))
            for term, doc_count in document_frequency.items()
        }

    def search(self, query, top_k, candidate_rows=None):
        """Return the row indices of the top_k passages most relevant to `query` by BM25 score.

        `candidate_rows` optionally restricts scoring to a subset of rows (used by metadata filtering).
        """
        # Keep only query words that occur somewhere in the corpus — the rest can never match.
        query_terms = [term for term in tokenize(query) if term in self.idf]
        candidate_indices = range(len(self.tokenized_docs)) if candidate_rows is None else candidate_rows

        scored_rows = []
        for row_index in candidate_indices:
            term_counts = self.term_counts_per_doc[row_index]
            doc_length = self.doc_lengths[row_index]
            score = 0.0
            for term in query_terms:
                term_count = term_counts.get(term, 0)
                if not term_count:
                    continue
                # Okapi BM25 per-term score: IDF × saturated term frequency ÷ length normalization.
                length_normalizer = self.k1 * (1 - self.b + self.b * doc_length / (self.avg_len or 1))
                score += self.idf[term] * (term_count * (self.k1 + 1)) / (term_count + length_normalizer)
            if score > 0:                      # a score of 0 means no query word matched — not a candidate
                scored_rows.append((row_index, score))
        scored_rows.sort(key=lambda row_and_score: row_and_score[1], reverse=True)
        return [row_index for row_index, _ in scored_rows[:top_k]]


bm25 = BM25([record['text'] for record in records])


def rows_allowed_by_filter(work_filter):
    """Resolve a work_id scope to the row indices it allows, or None to search the whole corpus.

    This is a *hard* pre-filter: rows outside the scope are removed before any ranking happens.
    """
    if not work_filter:
        return None
    allowed_works = set(work_filter)
    return [row_index for row_index, record in enumerate(records) if record['work_id'] in allowed_works]


def bm25_search(query_text, query_vector, top_k, work_filter=None):
    """Keyword arm: rank by literal word overlap (BM25). The query vector is ignored."""
    candidate_rows = rows_allowed_by_filter(work_filter)
    return bm25.search(query_text, top_k, candidate_rows=candidate_rows)


def dense_retrieve(query_text, query_vector, top_k, work_filter=None):
    """Dense/semantic arm: rank by vector nearness using the stage-04 index. The query text is ignored."""
    candidate_rows = rows_allowed_by_filter(work_filter)
    return dense_search(query_vector, top_k, candidate_rows=candidate_rows)


def reciprocal_rank_fusion(rankings, damping, top_k):
    """Fuse several ranked lists using position only: each row scores sum(1 / (damping + rank)).

    Fusing on rank (not raw score) means BM25's and cosine's incomparable score scales never get mixed.
    """
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for rank, row_index in enumerate(ranking):     # rank is 0-based; the +1 makes the first position rank 1
            fused_scores[row_index] += 1.0 / (damping + rank + 1)
    ranked_rows = sorted(fused_scores.items(), key=lambda row_and_score: row_and_score[1], reverse=True)
    return [row_index for row_index, _ in ranked_rows[:top_k]]


def hybrid_search(query_text, query_vector, top_k, work_filter=None):
    """Hybrid arm: run both keyword and dense over the same rows, then fuse their rankings with RRF."""
    candidate_rows = rows_allowed_by_filter(work_filter)
    dense_ranking = dense_search(query_vector, CANDIDATE_POOL, candidate_rows=candidate_rows)
    keyword_ranking = bm25.search(query_text, CANDIDATE_POOL, candidate_rows=candidate_rows)
    return reciprocal_rank_fusion([dense_ranking, keyword_ranking], RRF_K, top_k)


# The three methods compared head-to-head; each takes (query_text, query_vector, top_k, work_filter).
RETRIEVERS = {'bm25': bm25_search, 'dense': dense_retrieve, 'hybrid-rrf': hybrid_search}

# ---- embed the labeled questions with the hosted Azure deployment (keyless), cached on disk ----
# No offline fallback: if a question is not cached and Azure is unreachable, embed_query raises and stops the run.
config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']

CACHE_DIR = ARTIFACTS / 'embedding-cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
query_api_calls = 0


def embed_query(text):
    """Embed one query string, reusing notebooks 03/04's on-disk cache; call Azure only on a cache miss.

    The vector is truncated + L2-normalized to the stored dimension so it lives in the indexed vectors' space.
    """
    global query_api_calls
    cache_file = CACHE_DIR / (hashlib.sha256(f'{deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        query_api_calls += 1
    return l2_normalize(full_vector[:stored_dims])


# Embed every answer-key question once; the measurement loop reuses these vectors.
query_vectors = [embed_query(question['question']) for question, _, _ in answer_key]

print(f'BM25 index: {len(bm25.idf):,} distinct terms over {N:,} passages (avg length {bm25.avg_len:.0f} tokens).')
print(f'top-k={TOP_K}, RRF constant k={RRF_K}, candidate pool per method={CANDIDATE_POOL}.')
print(f'Registered retrievers: {", ".join(RETRIEVERS)}.')
print(f'Embedded {len(query_vectors)} question(s) for grading; {query_api_calls} Azure call(s) (rest from cache).')


## 4. Measure and Inspect

- **Five lenses:** retrieval quality (Recall@1, Recall@k, MRR), **empty-result rate**, **filter correctness**, **p95 latency**, and a per-category split so we can see *where* each method wins.
- **Recall@k / MRR — graded against the labels.** For each question the answer key holds the rows from its cited scene. **Recall@1** is how often a relevant row ranks first, **Recall@k** how often one lands in the top-k, and **MRR** rewards putting it higher (reciprocal of the first relevant rank). These are the same label-graded measures used upstream, now run through *real* retrieval.
- **Empty-result rate:** the fraction of queries a method returns **nothing** for. Dense never returns empty (every vector has a nearest neighbor); BM25 *can* when no query term appears in any passage — an honest, method-specific failure worth surfacing.
- **Filter correctness:** when a query names a scope, we apply the `work_id` filter and check two things — that **every** returned row actually lies in the allowed scope (the hard constraint is honored, a regression guard), and that filtering **does not drop recall** when the answer's work is in scope. A filter that silently removes the answer is the failure this catches.
- **p95 latency:** the 95th-percentile per-query time in milliseconds — the tail a user actually feels, not the average. At ten plays all methods are fast; hybrid runs two searches plus fusion, so it is the slowest, and we report the gap honestly.
- **Per-category split:** Recall@k broken out by question `category` (e.g. `exact_quote` vs. conceptual) — the clearest evidence for *why* hybrid is chosen: it should match the best single method on each category rather than winning on average while losing a category.
- **Honest limit:** the labeled dev set is small, so single percentages are coarse. This pre-screens retrieval methods; the end-to-end judges are generation (notebook 08) and continuous evaluation (notebook 10).


In [ ]:
K = TOP_K   # grade at the served top-k


def grade_ranking(ranked_rows, relevant_rows):
    """Score one ranked result list against the relevant rows: return (hit@1, hit@K, reciprocal_rank).

    `hit@1` is 1 if a relevant row is first, `hit@K` is 1 if one is in the top-K, and the reciprocal
    rank (1 / position) rewards placing the first relevant row higher.
    """
    first_relevant = next((position for position, row in enumerate(ranked_rows) if row in relevant_rows), None)
    if first_relevant is None:
        return 0, 0, 0.0
    return int(first_relevant == 0), int(first_relevant < K), 1.0 / (first_relevant + 1)


def evaluate_method(retriever):
    """Run one retrieval method over every answer-key question (corpus-wide, no filter) and summarize it.

    Returns Recall@1, Recall@K, MRR, empty-result rate, p95 latency, and a per-category Recall@K breakdown.
    """
    top1_hits = topk_hits = empty_results = 0
    reciprocal_rank_total = 0.0
    latencies_ms = []
    topk_by_category = defaultdict(lambda: [0, 0])   # category -> [topk_hits, question_count]
    for query_vector, (question, relevant_rows, _scope) in zip(query_vectors, answer_key):
        started = time.perf_counter()
        ranked_rows = retriever(question['question'], query_vector, TOP_K)
        latencies_ms.append((time.perf_counter() - started) * 1000)
        if not ranked_rows:
            empty_results += 1
        hit1, hitk, reciprocal_rank = grade_ranking(ranked_rows, relevant_rows)
        top1_hits += hit1
        topk_hits += hitk
        reciprocal_rank_total += reciprocal_rank
        category_bucket = topk_by_category[question.get('category', 'uncategorized')]
        category_bucket[0] += hitk
        category_bucket[1] += 1
    question_count = max(len(answer_key), 1)
    sorted_latencies = sorted(latencies_ms)
    p95_latency = sorted_latencies[min(len(sorted_latencies) - 1, int(0.95 * len(sorted_latencies)))] if sorted_latencies else 0.0
    return {
        'recall@1': top1_hits / question_count,
        f'recall@{K}': topk_hits / question_count,
        'mrr': reciprocal_rank_total / question_count,
        'empty_rate': empty_results / question_count,
        'p95_latency_ms': p95_latency,
        'by_category': {category: hits / count for category, (hits, count) in topk_by_category.items()},
    }


method_metrics = {name: evaluate_method(retriever) for name, retriever in RETRIEVERS.items()}

# ---- filter correctness: when a query names a scope, the hard constraint must be honored and keep the answer ----
filter_checks = {'scoped_questions': 0, 'constraint_honored': 0, 'recall_preserved': 0}
for query_vector, (question, relevant_rows, scope) in zip(query_vectors, answer_key):
    if not scope:
        continue
    filter_checks['scoped_questions'] += 1
    filtered = dense_retrieve(question['question'], query_vector, TOP_K, work_filter=scope)
    # 1) every returned row lies inside the allowed scope (the hard filter is actually applied)
    if all(records[row]['work_id'] in scope for row in filtered):
        filter_checks['constraint_honored'] += 1
    # 2) filtering did not drop the relevant row when the answer's work is in scope
    if any(row in relevant_rows for row in filtered):
        filter_checks['recall_preserved'] += 1

scoped = max(filter_checks['scoped_questions'], 1)

categories = sorted({question.get('category', 'uncategorized') for question, _, _ in answer_key})
print(f'Measured over {len(answer_key)} resolved dev question(s); graded at top-{K}.\n')
header = ('method', 'recall@1', f'recall@{K}', 'mrr', 'empty%', 'p95 ms')
print('{:<12} {:>9} {:>9} {:>7} {:>7} {:>8}'.format(*header))
for name, m in method_metrics.items():
    print('{:<12} {:>8.0%} {:>8.0%} {:>7.3f} {:>6.0%} {:>8.2f}'.format(
        name, m['recall@1'], m[f'recall@{K}'], m['mrr'], m['empty_rate'], m['p95_latency_ms']))

print(f'\nRecall@{K} by question category:')
print('{:<12} '.format('method') + ' '.join(f'{cat:>16}' for cat in categories))
for name, m in method_metrics.items():
    cells = ' '.join(f'{m["by_category"].get(cat, 0.0):>15.0%} ' for cat in categories)
    print('{:<12} '.format(name) + cells)

print(f'\nFilter demo ({filter_checks["scoped_questions"]} scoped question(s)): '
      f"constraint honored {filter_checks['constraint_honored']}/{scoped}, "
      f"relevant row kept {filter_checks['recall_preserved']}/{scoped}.")

# Inspect one query's top result under the hybrid method, by eye.
if answer_key:
    sample_question, sample_relevant, _ = answer_key[0]
    sample_ranked = hybrid_search(sample_question['question'], query_vectors[0], TOP_K)
    if sample_ranked:
        top_record = records[sample_ranked[0]]
        verdict = 'RELEVANT' if sample_ranked[0] in sample_relevant else 'off-target'
        print(f'\nSample hybrid top-1 for "{sample_question["question"][:70]}..."')
        print(f"  [{verdict}] {top_record['work']} {top_record['scenes']} "
              f"speakers={', '.join(top_record['speakers']) or 'none'}")
        print('  ' + top_record['text'][:240].replace('\n', ' ') + ('…' if len(top_record['text']) > 240 else ''))


### Metadata filtering, generalized (work · scene · speaker)

Section 4 measured the `work_id` filter. Real systems filter on more than the document — author, section, date, speaker — so here we **enable and demonstrate** the same **hard AND filter** generalized to the other metadata every row already carries (`scenes`, `speakers`).

- **What it does:** `filter_rows` keeps only rows that satisfy *all* given constraints; any retriever then ranks within that smaller pool. We demo it with **BM25** so it runs fully offline (no query embedding needed), but the same `filter_rows` output works for `dense` and `hybrid` too.
- **Why it matters:** scoping raises precision and cuts cost when the query names a scope ("in *Macbeth*", "Act 1, Scene 1", "lines spoken by Macbeth"), and it composes with keyword, dense, or hybrid search alike — filtering is a *constraint layer*, not a competing ranker.
- **Production note:** locally we can filter on any stored field; in **Azure AI Search** a field must be declared **filterable** in the index schema to filter on it efficiently. Our stage-04 schema marks only `work_id` filterable today — promote `scenes`/`speakers` to the filterable set there to enable these scopes in production.


In [ ]:
def filter_rows(works=None, scene=None, speakers=None):
    """Return the row indices whose metadata satisfies ALL given constraints (a hard AND pre-filter).

    works: iterable of work_id to keep; scene: an (act, scene) tuple the row must contain;
    speakers: iterable of speaker names, kept if the row has ANY of them. A None constraint is ignored.
    """
    allowed_works = set(works) if works else None
    wanted_speakers = set(speakers) if speakers else None
    selected_rows = []
    for row_index, record in enumerate(records):
        if allowed_works is not None and record['work_id'] not in allowed_works:
            continue
        if scene is not None and not any(tuple(pair) == scene for pair in record['scenes']):
            continue
        if wanted_speakers is not None and wanted_speakers.isdisjoint(record['speakers']):
            continue
        selected_rows.append(row_index)
    return selected_rows


def show_keyword_search(query, works=None, scene=None, speakers=None, show=3):
    """Run keyword (BM25) search under a metadata filter and print the top hits — fully offline (no embedding)."""
    scope_rows = filter_rows(works=works, scene=scene, speakers=speakers)
    hits = bm25.search(query, show, candidate_rows=scope_rows)
    scope_label = ' · '.join(part for part in (
        f"work={','.join(works)}" if works else '',
        f"scene={scene}" if scene else '',
        f"speaker={','.join(speakers)}" if speakers else '',
    ) if part) or 'whole corpus'
    print(f'query="{query}"  |  filter: {scope_label}  |  candidate rows: {len(scope_rows):,}')
    for row_index in hits:
        record = records[row_index]
        print(f"   {record['work']} {record['scenes']} speakers={record['speakers']}")
    assert all(row in set(scope_rows) for row in hits)   # every hit must honor the hard constraints
    return hits


# Resolve concrete, real demo values from the data so nothing is hardcoded to a missing scene/speaker.
DEMO_QUERY = 'blood'
title_to_work_id = {record['work']: record['work_id'] for record in records}
macbeth_id = title_to_work_id['Macbeth']
macbeth_rows = filter_rows(works=[macbeth_id])
macbeth_speaker = 'MACBETH' if any('MACBETH' in records[i]['speakers'] for i in macbeth_rows) else records[macbeth_rows[0]]['speakers'][0]
# Use the scene of the top Macbeth hit for the query, so the scene-scoped demo actually returns that passage.
macbeth_top_hit = bm25.search(DEMO_QUERY, 1, candidate_rows=macbeth_rows)
macbeth_scene = tuple(records[macbeth_top_hit[0]]['scenes'][0]) if macbeth_top_hit else tuple(records[macbeth_rows[0]]['scenes'][0])

print('1) Same keyword query, whole corpus vs. scoped to one play:')
show_keyword_search(DEMO_QUERY)
show_keyword_search(DEMO_QUERY, works=[macbeth_id])

print(f'\n2) Scope to a single scene {macbeth_scene} of Macbeth:')
show_keyword_search(DEMO_QUERY, works=[macbeth_id], scene=macbeth_scene)

print(f'\n3) Scope to lines spoken by {macbeth_speaker}:')
show_keyword_search(DEMO_QUERY, works=[macbeth_id], speakers=[macbeth_speaker])

print('\nThe filter is a hard AND gate: each added constraint shrinks the candidate pool before ranking.')


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** prefer the method with the best **category floor** (highest *worst-category* Recall@k) — an evidence desk must not collapse on any question type; among equally-robust methods, prefer the one that ranks the relevant passage **highest** (**MRR**, then Recall@k); break remaining ties by **lower p95 latency**, then by a simplicity order. The rule is applied to the measured numbers, not asserted.
- **Why the category floor gates first:** the Evidence Desk serves *both* exact-quote and conceptual questions. A method that wins on average while scoring zero on, say, character-lookup questions is useless here. BM25 is the usual casualty — strong on exact quotes, blind to vocabulary mismatch — so it is rejected on its weak floor before quality is even considered.
- **Honest, data-dependent outcome (no pre-ordained winner):** on this small corpus **dense** and **hybrid** both keep every category's floor high, so the tie-break falls to MRR. Dense often ranks the single best passage at #1, whereas RRF can *demote* a decisively-correct exact-quote hit to #2 by averaging it with a weaker dense neighbour — so on this set the rule may well select **dense** outright. That is the honest result, reported rather than massaged toward a predetermined "hybrid wins."
- **Hybrid stays the production-robust default — and is the production target.** RRF's value is a better *floor* as the query mix broadens and one method starts failing cases the other catches; on a tiny, clean labeled set that advantage may not yet show. The handoff therefore records the measured local winner *and* names Azure AI Search's hybrid query as the target to re-confirm end-to-end (notebooks 08, 10) — exactly as notebook 04 selected local IVF while naming HNSW in production.
- **top-k and filters are settled separately:** `top-k` is fixed at the served value (retuned once 06/08 can measure it), and the metadata filter is demonstrated and checked (constraint honored, recall preserved) rather than swept as a competing arm.
- **Expected result:** the selected method with the numbers that justify it (worst-category Recall@k, MRR, Recall@k, p95 latency), and the rejected methods noted with their measured reason.


In [ ]:
SIMPLICITY_ORDER = {'bm25': 0, 'dense': 1, 'hybrid-rrf': 2}   # last-resort tie-break; lower = simpler


def worst_category(metrics):
    # The method's floor across question categories — the weakest category it serves.
    scores = metrics['by_category'].values()
    return min(scores) if scores else 0.0


def selection_key(name):
    m = method_metrics[name]
    # Robustness first (highest category floor), then rank quality (MRR, then Recall@k),
    # then lower p95 latency, then the simpler method. Negatives so `min` prefers higher-is-better.
    return (-worst_category(m), -m['mrr'], -m[f'recall@{K}'], m['p95_latency_ms'], SIMPLICITY_ORDER.get(name, 99))


selected_method = min(method_metrics, key=selection_key)
selected = method_metrics[selected_method]
rejected = [name for name in method_metrics if name != selected_method]

print(f'Selected method: {selected_method} (top-k={TOP_K}, RRF k={RRF_K}, metric={metric})')
print(f"  recall@1 / recall@{K} / mrr: {selected['recall@1']:.0%} / {selected[f'recall@{K}']:.0%} / {selected['mrr']:.3f}")
print(f"  worst-category recall@{K}:  {worst_category(selected):.0%}  (robust across categories)")
print(f"  empty-result rate:          {selected['empty_rate']:.0%}")
print(f"  p95 latency:                {selected['p95_latency_ms']:.2f} ms/query")
print('\nRejected:')
for name in rejected:
    m = method_metrics[name]
    if worst_category(m) < worst_category(selected):
        reason = 'weaker category floor (regresses on a category)'
    elif m['mrr'] < selected['mrr']:
        reason = 'ranks the relevant passage lower (lower MRR)'
    else:
        reason = 'higher latency / less simple'
    print(f"  {name}: recall@{K}={m[f'recall@{K}']:.0%} mrr={m['mrr']:.3f} "
          f"worst-cat={worst_category(m):.0%} p95={m['p95_latency_ms']:.2f}ms — {reason}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/retrieval-handoff.json` — the decision record notebook 06 loads: the selected method (`bm25` / `dense` / `hybrid-rrf`), the served `top-k`, the RRF constant, the distance metric, the fixed index reference (type + ANN params), the metadata filter policy, the full candidate comparison (quality, per-category, empty-rate, p95 latency), the filter-correctness results, and a back-reference to the stage-04 index.
- **Why a config handoff, not a data dump:** the index (`index.json`) already holds the rows and vectors; retrieval adds a *decision* — how to query that index — so the handoff is the reproducible configuration plus the evidence for it. Notebook 06 reconstructs the exact candidate lists by replaying the selected method over the fixed index.
- **Provenance preserved:** nothing about the rows changes here; candidates still carry `chunk_id`, work, scenes, and speakers, so reranking, context building, and citations downstream resolve back to the source lines.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff.
- **Expected result:** the handoff written, with the selected method, top-k, metric, and index reference printed.


In [ ]:
retrieval_handoff_path = ARTIFACTS / 'retrieval-handoff.json'

# retrieval-handoff.json: the decision record notebook 06 replays — the selected query strategy over the
# fixed stage-04 index, plus the measured comparison that justified it. The index (index.json) holds the data.
retrieval_handoff = {
    'stage': '05-retrieval',
    'selected_method': selected_method,            # bm25 | dense | hybrid-rrf
    'top_k': TOP_K,
    'rrf_k': RRF_K if selected_method == 'hybrid-rrf' else None,
    'metric': metric,
    'bm25_params': {'k1': bm25.k1, 'b': bm25.b},
    'filter_policy': {'field': 'work_id', 'kind': 'hard_prefilter', 'demonstrated': True},
    'index_reference': {                            # the fixed upstream this strategy queries
        'stage': indexing_handoff.get('stage', '04-indexing'),
        'index_type': index_type,
        'params': index_payload.get('params', {}),
        'dims': stored_dims,
        'embedding_backend': embedding_backend,
    },
    'selection_rule': 'worst-category recall@k, then mrr, then recall@k, then p95 latency, then simplicity',
    'candidates': method_metrics,                   # quality, per-category, empty-rate, p95 for every method
    'filter_checks': filter_checks,                 # scoped questions, constraint honored, recall preserved
    'resolved_questions': len(answer_key),
    'production_target': {'service': 'azure-ai-search', 'query': 'hybrid (keyword + vector) with filters'},
}
retrieval_handoff_path.write_text(json.dumps(retrieval_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote decision record to {retrieval_handoff_path.relative_to(ROOT)}')
print(f'Selected: {selected_method} | top-k={TOP_K} | metric={metric} | '
      f"index={index_type}{index_payload.get('params', {})}")
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit — do not commit the handoff.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **Retrieval is the first stage whose metric grades against the labels directly.** The offline stages measured proxies (size distribution, recall-vs-exact); here Recall@k and MRR compare retrieved rows to the *cited* scene. That is a stronger signal — but still a proxy for answer quality, because a retrieved passage can be relevant and still not get used or cited correctly downstream (notebooks 08, 10).

- **The three methods fail on different queries — that is the point, not a defect.** BM25 wins exact-quote questions and whiffs on paraphrase; dense wins concepts and can miss a rare literal. The per-category split is where this shows, and it is why the selection gates on the *category floor* first — an evidence desk cannot afford a method that zeroes out on one question type.

- **Hybrid did not automatically win here — and we didn't force it.** On this small, clean labeled set a single strong method (dense) already holds every category's floor, and RRF can even *lower* MRR by demoting a decisively-correct exact-quote hit to #2. So the deterministic rule may select dense outright. Hybrid's value — a better floor once one method starts failing cases the other catches — is latent at this scale, not yet earned. We record it as the production-robust default to re-confirm as the query mix broadens, rather than pretending the tiny set already proves it.

- **RRF fuses on rank for a reason.** BM25 and cosine scores are not comparable, so combining raw scores needs fragile normalization. Rank fusion needs none, which is why hybrid uses a standard constant rather than tuned per-corpus weights. If a future corpus makes one method clearly dominant, revisit weighted fusion then — with measurements.

- **Filters are powerful and dangerous.** A correct scope raises precision for free; a wrong scope silently deletes the answer. We measured that the hard constraint is honored and that filtering keeps the relevant row — a regression guard — rather than assuming filters are safe. Production should derive the scope carefully (often in query handling, notebook 09), not from brittle string matching.

- **At ten plays, latency differences are tiny.** Hybrid runs two searches plus fusion and is the slowest, but the absolute p95 gap is small. The honest read is that *method robustness*, not speed, drives the choice at this scale; the latency tradeoff becomes real only as the corpus and query rate grow, where the production hybrid query (Azure AI Search) is measured on its own terms.

- **How we will revisit.** Retrieval is a swappable layer: change the method, top-k, or filter policy, re-export the handoff, and re-run 06→08 on the **same** questions. Iterate on the **development** set; check the **held-out** set only periodically so the evaluation stays honest. Retrieval and chunking are the two highest-leverage knobs to retune once generation quality is measurable.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 04 artifacts, not hidden state. BM25 and fusion are pure local compute; only question embedding can touch Azure, and it is served from the 03/04 cache when available.
- **What we achieved:** three retrieval methods (BM25, dense, hybrid/RRF) run over the fixed stage-04 index and graded against the labeled questions on Recall@1/@k, MRR, empty-result rate, and p95 latency; a per-category split showing where each method wins; a metadata filter demonstrated and checked (constraint honored, recall preserved); a deterministic, robustness-first selection; and the exported retrieval handoff.
- **Honest limits:** the labeled dev set is small, so single percentages are coarse and the methods can tie at the top; the selection then prefers hybrid for category robustness. The local BM25/RRF are faithful stand-ins for the production Azure AI Search hybrid query. The real judges are generation (notebook 08) and continuous evaluation (notebook 10).
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff.
- **Next stage — 06 Reranking:** load `.artifacts/retrieval-handoff.json`, replay the selected method over the fixed index to produce a candidate set, then compare no-reranking vs. semantic vs. bounded LLM reranking on nDCG@k and evidence coverage — keeping a reranking step only if it beats retrieval's own order by enough to justify its latency and cost.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. Why not just use dense retrieval — isn't semantic search strictly better than keywords?**
No. Dense search matches *meaning*, which is great for paraphrase and concept questions, but it can miss an exact rare token the query hinges on — a proper name ("Birnam"), an archaic word ("hebenon"), a precise quote. BM25 nails those and fails the opposite way (it can't link "killed" to "slew"). They fail on *different* queries, so for a mixed query load neither dominates in general. On a small, clean labeled set, though, one method can look strictly best — which is why we measure rather than assume.

**Q2. Why does RRF fuse on *rank* instead of combining the actual scores?**
Because BM25 scores and cosine similarities live on incomparable scales — BM25 is unbounded and corpus-dependent, cosine sits in $[-1, 1]$. Adding or averaging them requires fragile per-corpus normalization that breaks when the corpus changes. RRF reads only each passage's *position* in each list, so a passage ranked #2 by BM25 and #3 by dense gets a principled fused score with **no tuning**. The constant $k$ (60) just damps the influence of the very top ranks so a strong-in-both passage can beat a first-in-only-one. That robustness is why RRF is a common production default — though the same averaging can *demote* a case where one method is decisively right, which is exactly why hybrid doesn't always top a tiny labeled set.

**Q3. How is top-k different from the ANN `nprobe` knob in notebook 04?**
They operate at different layers. `nprobe` is an *index* knob — how many clusters the ANN search opens internally before it even has a ranked list; it trades index fidelity for speed. `top-k` is a *retrieval* knob — how many of the finally-ranked candidates we hand to the next stage; it trades recall against downstream noise and cost. You can change top-k without touching the index, and vice versa.

**Q4. A filter looks like free precision — when does it hurt?**
Only when the scope is wrong. A filter is a *hard* constraint: it deletes every row outside the scope before ranking. If the query's scope is correct ("only *Hamlet*"), it raises precision and cuts cost. If the scope is wrong or too narrow, it can remove the very passage that answers the question, dropping recall to zero for that query with no error raised. That is why we measure *filter correctness* — both that the constraint is honored and that it keeps the relevant row — rather than assuming filters are always safe.

**Q5. Retrieval scored well — does that mean the final answer will be correct?**
Not on its own. High Recall@k means the right passage is *somewhere* in the candidate set — necessary, but not sufficient. The passage still has to survive reranking (06), fit the context budget (07), and actually be used and cited faithfully by the generator (08). Retrieval sets the ceiling; the later stages decide how much of it is realized. The honest end-to-end judges are notebooks 08 and 10.

**Q6. Where do query rewriting and reranking fit, and why aren't they here?**
Both wrap retrieval but are separate stages. **Query rewriting** (notebook 09) changes the *query* before searching — rewriting, decomposition, multi-query, follow-up resolution — and is kept out so this stage measures retrieval on the question *as written*. **Reranking** (notebook 06) reorders the *fixed* candidate set this stage returns, using a more expensive model. Keeping the boundaries clean means each lever's contribution is measured alone instead of being tangled together.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/retrieval-handoff.json` — the selected method, the served top-k, the metric, the fixed index reference, the filter policy, and the full measured comparison that justified the choice. Notebook 06 (Reranking) replays this configuration over the fixed index to generate a candidate set, then measures whether reordering it with a stronger model improves nDCG and evidence coverage enough to justify its latency and cost.
